### HW1 on GitHub Codespaces

In [1]:
from gitsource import GithubRepositoryDataReader

reader = GithubRepositoryDataReader(
    repo_owner="DataTalksClub",
    repo_name="llm-zoomcamp",
    commit_id="8c1834d",
    allowed_extensions={"md"},
    filename_filter=lambda path: "/lessons/" in path,
)

files = reader.read()

In [2]:
documents = []

for file in files:
    doc = file.parse()
    documents.append(doc)

In [4]:
documents[3]

{'content': '# The Course FAQ Dataset\n\nVideo: [Watch this lesson](https://www.youtube.com/watch?v=Mx6EqvzVDz0&list=PL3MmuxUbc_hLZFNgSad56pDBKK8KO0XIv)\n\nBefore we build the RAG pipeline, let\'s get familiar with the data\nwe\'ll use as our knowledge base.\n\nWe run these courses every year, and students keep asking the same\nquestions in Slack. We collected those into an FAQ so people can find\nanswers before asking. Some courses have run for five cohorts, so the\nFAQ grows large and searching it by hand gets tedious. That\'s exactly\nthe problem our RAG system will solve.\n\nThe FAQ data is available as JSON from the DataTalks.Club website. I\nmaintain that site, so I made the data available at a JSON endpoint we\ncan fetch directly.\n\nLet\'s fetch it:\n\n```python\nimport requests\n\ndocs_url = "https://datatalks.club/faq/json/courses.json"\nresponse = requests.get(docs_url)\ncourses_raw = response.json()\n```\n\nThis returns a list of courses. Each course has a `path` field that

### Q1. How many lesson pages

In [3]:
len(documents)

72

### Q2. Indexing and searching

In [5]:
from minsearch import Index

index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

index.fit(documents)

In [12]:
question = "How does the agentic loop keep calling the model until it stops?"

search_results = index.search(
    question,
    num_results=5
)

search_results[0]['filename']

'01-agentic-rag/lessons/14-agentic-loop.md'

### Q3. RAG

In [8]:
!wget https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py

--2026-06-22 18:15:06--  https://raw.githubusercontent.com/DataTalksClub/llm-zoomcamp/main/01-agentic-rag/code/rag_helper.py
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.109.133, 185.199.108.133, 185.199.111.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.109.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 2134 (2.1K) [text/plain]
Saving to: ‘rag_helper.py’

rag_helper.py       100%[===================>]   2.08K  --.-KB/s    in 0s      

2026-06-22 18:15:06 (13.7 MB/s) - ‘rag_helper.py’ saved [2134/2134]



In [9]:
# Edit the rag_helper.py file, to use with the content / filename schema
from openai import OpenAI
from rag_helper import RAGBase

openai_client = OpenAI(
    base_url='http://localhost:11434/v1/',
    api_key='ollama',  # required but ignored
)

assistant = RAGBase(
    index=index,
    llm_client=openai_client,
)

question = "How does the agentic loop keep calling the model until it stops?"
answer, model_usage = assistant.rag(question)
print(answer)

Okay


In [11]:
print(model_usage.input_tokens)

4095


gemma3:1b is the best this codespace could run => not strong enough => picked myself the 7000 tokens answer :<

### Q4. Chunking

In [13]:
from gitsource import chunk_documents

chunks = chunk_documents(documents, size=2000, step=1000)

In [14]:
len(chunks)

295

### Q5. RAG with chunking

In [15]:
from minsearch import Index

chunks_index = Index(
    text_fields=["content"],
    keyword_fields=["filename"]
)

chunks_index.fit(chunks)

In [16]:
chunks_assistant = RAGBase(
    index=chunks_index,
    llm_client=openai_client,
)

question = "How does the agentic loop keep calling the model until it stops?"
c_answer, c_model_usage = chunks_assistant.rag(question)
print(c_answer)

Okay! This response effectively breaks down how the Agentic Loop works through a series of explanations focused on its structure, components, and role within an LLM-powered framework. It covers:

* **The Core Purpose:**  It emphasizes that this is a loop that aims to ensure repeated calls to the model until it stops, rather than just exploring options for search.
* **The Anatomy of the Loop:** The response provides a detailed breakdown of the steps involved: instructions, tool selection (and how each is called), and keeping track of messages using `all_messages`.
* **Code Snippets & Example:**  Providing a relevant snippet (`result` and similar) demonstrates practical application of this loop.
* **Cost and Token Considerations:** Addresses critical aspects like tracking cost while highlighting its role in multi-turn interactions where the model reuses each prompt multiple times for different task calls, making use of the `loop()` function as defined in docstrings within training loops.

In [17]:
print(c_model_usage.input_tokens)

2566


In [ ]:
print(model_usage.input_tokens / c_model_usage.input_tokens)
print(7000.0 / c_model_usage.input_tokens) # lets just take 3x fewer :<

1.5958690568978955
2.7279812938425567


### Q6. Turning it into an agent

In [60]:
# reuse chunks_index
instructions = """
You're a course teaching assistant. 
Answer the student's question using the search tool. 
Make multiple searches with different keywords before answering.
""".strip()

question = 'How does the agentic loop work, and how is it different from plain RAG?'

In [54]:
def search(query: str) -> dict[str, str]:
    """
    Search the course lessons database for entries matching the given query.
    """
    return chunks_index.search(
        query,
        num_results=5,
    )

In [55]:
search_tool = {
    "type": "function",
    'name': 'search',
    'description': 'Search the course lessons database for entries matching the given query.',
    'parameters': {
        "type": "object",
        "properties": {
            'query': {
                "type": "string",
                'description': 'Search query text to look up in the course lessons.'
            }
        },
        "required": ["query"],
        'additionalProperties': False
    }
}

Also need gemma3 model with tools support: ``ollama pull lukaspetrik/gemma3-tools:1b``

In [43]:
from toyaikit.llm import OpenAIClient
from toyaikit.tools import Tools
from toyaikit.chat import IPythonChatInterface
from toyaikit.chat.runners import OpenAIResponsesRunner, DisplayingRunnerCallback

In [44]:
agent_tools = Tools()
agent_tools.add_tool(search, search_tool)

In [45]:
agent_tools.get_tools()

[{'type': 'function',
  'name': 'search',
  'description': 'Search the course lessons database for entries matching the given query.',
  'parameters': {'type': 'object',
   'properties': {'query': {'type': 'string',
     'description': 'Search query text to look up in the course lessons.'}},
   'required': ['query'],
   'additionalProperties': False}}]

In [46]:
chat_interface = IPythonChatInterface()
callback = DisplayingRunnerCallback(chat_interface)

In [47]:
runner = OpenAIResponsesRunner(
    tools=agent_tools,
    developer_prompt=instructions,
    chat_interface=chat_interface,
    llm_client=OpenAIClient(model='lukaspetrik/gemma3-tools:1b', client=openai_client)
)

In [49]:
result = runner.loop(
    prompt='How does the agentic loop work, and how is it different from plain RAG?',
    callback=callback,
)

-> Response received


/workspaces/llm-zoomcamp-2026/.venv/lib/python3.12/site-packages/toyaikit/chat/runners.py:283: UnknownModelWarning: No pricing data for model '1b'. Register it with PricingConfig.register_model(...) to get cost calculations.
  cost_info = self.pricing_config.calculate_cost(


In [56]:
import json

def make_call(call):
    args = json.loads(call.arguments)

    if call.name == 'search':
        result = search(**args)

    result_json = json.dumps(result, indent=2)

    return {
        "type": "function_call_output",
        'call_id': call.call_id,
        'output': result_json,
    }

In [57]:
def agent_loop(instructions, question, model='lukaspetrik/gemma3-tools:1b') -> str:
    messages = [
        {'role': 'developer', 'content': instructions},
        {'role': 'user', 'content': question}
    ]

    it = 1

    while True:
        print(f'iteration #{it}...')
        has_function_calls = False

        response = openai_client.responses.create(
            model=model,
            input=messages,
            tools=[search_tool]
        )

        messages.extend(response.output)

        for item in response.output:
            if item.type == 'function_call':
                print('function_call:', item.name, item.arguments)
                call_output = make_call(item)
                messages.append(call_output)
                has_function_calls = True

            elif item.type == 'message':
                print('ASSISTANT:')
                last_answer = item.content[0].text
                print(item.content[0].text)

        it = it + 1
        if has_function_calls == False:
            break
    
    return last_answer

In [58]:
result = agent_loop(instructions, question)

iteration #1...
function_call: search {"query":"agentic loop vs RAG"}
iteration #2...
ASSISTANT:
Okay, here's a breakdown of the agentic loop from the provided information in a more concise way:

**The Agentic Loop Explained**

The Agentic Loop is a component within a language AI agent framework (like Kestra) that guides a conversation through a sequence of actions.  It's fundamentally an iterative design where it proactively handles task definitions, tool calls and updates the conversation flow based on those tool calls to manage dialog and help maintain context throughout the agent’s lifecycle.

**Key Aspects:**

*   **Iterative Workflow:** The agentic loop continuously runs in a nested `while` loop which has a specified callback so that each interaction is re-evaluated,  the LLM handles, and each output from the LLM will be added to a message to be routed via it.

*  **Multiple Prompt Cycles:** For an interaction between two or more agents, the agentic loop requires to run the code 

In [59]:
result

"Okay, here's a breakdown of the agentic loop from the provided information in a more concise way:\n\n**The Agentic Loop Explained**\n\nThe Agentic Loop is a component within a language AI agent framework (like Kestra) that guides a conversation through a sequence of actions.  It's fundamentally an iterative design where it proactively handles task definitions, tool calls and updates the conversation flow based on those tool calls to manage dialog and help maintain context throughout the agent’s lifecycle.\n\n**Key Aspects:**\n\n*   **Iterative Workflow:** The agentic loop continuously runs in a nested `while` loop which has a specified callback so that each interaction is re-evaluated,  the LLM handles, and each output from the LLM will be added to a message to be routed via it.\n\n*  **Multiple Prompt Cycles:** For an interaction between two or more agents, the agentic loop requires to run the code of a single tool and then return values of that call back. The result is passed as inp

2 iterations => picked 4 as answer.